# 2.3 — Procesamiento: índice de marginación (censal → mensual)

**Capa 1 del protocolo.** La marginación solo se mide en los censos (2010, 2015 y 2020), pero el panel necesita un valor **por mes**. Esta libreta revisa los archivos municipales de `01_ingesta/1.3-indice_-marginacion.ipynb` y la interpolación mensual que usa el panel.

Diccionario de datos: `references/marginacion/data_dictionary_conapo_IM.md`.

| | |
|---|---|
| **Entrada** | `data/interim/marginacion/marginacion_municipal_{2010,2015,2020}.csv` (72 municipios de Sonora por año, CONAPO) |
| **Salida (opcional)** | `data/interim/marginacion/marginacion_hermosillo_mensual.csv` |

## Checklist

- [x] Mismas columnas en los tres años.
- [x] Una fila de Hermosillo (`26030`) en cada año.
- [x] Revisar qué indicadores son comparables entre años (`IM` sí, `IMN` no → se elimina).
- [x] Interpolación lineal 2015 → 2020; valor constante desde ene-2020.
- [ ] (Opcional) Usar 2010 si se amplía la ventana del panel (`PANEL_START_YEAR`).
- [ ] AGEB urbana (IMU 2020): pendiente de la cartografía INEGI.

### Ideas clave

- **CONAPO** calcula el índice de marginación con 9 indicadores de carencias (educación, vivienda, ingresos). Se usa la serie **DP2**, la única comparable entre 2010, 2015 y 2020.
- En esta serie, un `IM` **más alto** parece indicar **menos** marginación (ver notas del diccionario).
- **Interpolar** es estimar los meses entre dos censos trazando una línea recta entre ambos valores.

### Columnas de los archivos

| Columna | Qué es |
|---------|--------|
| `ANALF` | % de 15 años o más analfabeta |
| `SBASC` | % de 15 años o más sin educación básica |
| `OVSDE`, `OVSEE`, `OVSAE`, `OVPT` | % de ocupantes en viviendas sin drenaje, sin luz, sin agua entubada o con piso de tierra |
| `VHAC` | % de viviendas con hacinamiento |
| `PL.5000` | % de población en localidades de menos de 5,000 habitantes |
| `PO2SM` | % de población ocupada que gana hasta 2 salarios mínimos |
| `IM` / `GM` | Índice de marginación / grado (Muy bajo … Muy alto) |
| `IMN` | Índice normalizado (0–1). **Se elimina** del panel porque no es comparable entre años (ver la verificación) |

### Funciones que se usan

| De dónde | Nombre | Qué hace |
|----------|--------|----------|
| `config.py` | `INTERIM_DATA_DIR`, `PANEL_START_YEAR`, `PANEL_END_YEAR` | Carpeta `data/interim` y años del panel (2015–2020) |
| `panel.py` | `marginacion_hermosillo()` | Lee los censos 2015 y 2020, toma la fila de Hermosillo e interpola cada indicador mes a mes (sin `IMN`). **Es lo que entra al panel.** |
| este notebook | `sin_sufijo()`, `fila_hermosillo()` | Repiten pasos de `marginacion_hermosillo()` para poder revisarlos aquí |

## Configuración e imports

Se cargan las librerías, las constantes de `config.py` y la función de `panel.py`.

**Si falla** con `No module named 'sonora_health_climate_eda'`, el kernel elegido no es `.venv`.

In [ ]:
from pathlib import Path  # Path: herramienta para manejar rutas de carpetas y archivos
import sys  # sys: permite cambiar dónde busca módulos Python

# Este notebook vive en notebooks/02_procesamiento/: subimos dos carpetas hasta la raíz del proyecto
ROOT_DIR = Path.cwd().parent.parent  # notebooks/<seccion>/ -> raiz del proyecto
# Agrega la raíz a la lista de búsqueda de Python para poder importar el paquete del proyecto
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import pandas as pd  # pandas: trabajar con tablas (DataFrames)
import matplotlib.pyplot as plt  # matplotlib: hacer gráficas

# De config.py: carpeta data/interim y años del panel (2015 y 2020)
from sonora_health_climate_eda.config import INTERIM_DATA_DIR, PANEL_END_YEAR, PANEL_START_YEAR, PROCESSED_DATA_DIR
# De panel.py: la función que interpola la marginación de Hermosillo por mes
from sonora_health_climate_eda.panel import marginacion_hermosillo

MARG_DIR = INTERIM_DATA_DIR / "marginacion"  # carpeta data/interim/marginacion
ANIOS_CENSO = (2010, 2015, 2020)  # años en los que CONAPO publicó el índice
print("Carpeta de marginación:", MARG_DIR.relative_to(ROOT_DIR))

## Carga de los censos

Se lee el CSV de cada año y se guarda en un diccionario `censos` (año → tabla).

**Resultado:** los tres años tienen **72 filas** (los municipios de Sonora) y **17 columnas**.

In [ ]:
censos = {}  # diccionario vacío: aquí se guardará una tabla por año
for anio in ANIOS_CENSO:  # repite para 2010, 2015 y 2020
    path = MARG_DIR / f"marginacion_municipal_{anio}.csv"  # ruta del archivo de ese año
    if path.exists():  # solo lo lee si el archivo existe
        censos[anio] = pd.read_csv(path)  # lee el CSV y lo guarda con su año como llave
        print(anio, censos[anio].shape)  # shape = (filas, columnas)
    else:
        print(anio, "no encontrado:", path.relative_to(ROOT_DIR))

## Homologación de columnas entre años

En los Excel originales algunas columnas traen el año en el nombre (`IM_2020`). `sin_sufijo()` lo quita (`IM_2020` → `IM`) para que los tres años tengan las mismas columnas; después se compara qué columnas hay en cada año.

**Resultado:** las 17 columnas son **iguales** en los tres años y ninguna queda sola en un año. El notebook 1.3 ya había quitado los sufijos, así que aquí solo se confirma.

In [ ]:
def sin_sufijo(df: pd.DataFrame, anio: int) -> pd.DataFrame:
    # IM_2015 -> IM, GM_2015 -> GM, etc. (misma regla que panel.py).
    # rsplit("_", 1)[0] corta el nombre en el último "_" y se queda con la parte izquierda.
    return df.rename(columns={c: c.rsplit("_", 1)[0] for c in df.columns if c.endswith(f"_{anio}")})


# Aplica sin_sufijo a la tabla de cada año (diccionario año -> tabla homologada)
homologados = {anio: sin_sufijo(df, anio) for anio, df in censos.items()}
# Conjunto (set) de nombres de columnas de cada año
columnas = {anio: set(df.columns) for anio, df in homologados.items()}
# set.intersection = columnas que están en TODOS los años
comunes = set.intersection(*columnas.values()) if columnas else set()
print("Columnas comunes:", sorted(comunes))
for anio, cols in columnas.items():
    # cols - comunes = columnas que solo tiene ese año (lo ideal es una lista vacía)
    print(f"{anio} — solo en este año:", sorted(cols - comunes))

## Hermosillo en cada censo

`fila_hermosillo()` se queda con la fila cuya clave termina en `030` (Hermosillo es `26030`). Como los archivos solo traen municipios de Sonora, esa regla no confunde a Hermosillo con otro municipio.

**Resultado:** una fila por año, siempre con grado **Muy bajo**. Entre 2015 y 2020 el `IM` sube de 51.5 a 60.4 y la mayoría de las carencias bajan; en la siguiente celda se ve cuáles no.

In [ ]:
def fila_hermosillo(df: pd.DataFrame) -> pd.DataFrame:
    # Usa CVE_MUN si existe; si no, CVE_CONCATENADA (los archivos viejos la llamaban así)
    cve = "CVE_MUN" if "CVE_MUN" in df.columns else "CVE_CONCATENADA"
    # astype(str) convierte la clave a texto; str.endswith("030") busca las que terminan en 030
    return df[df[cve].astype(str).str.endswith("030")]  # misma regla que panel.py


# Fila de Hermosillo de cada año
hmo = {anio: fila_hermosillo(df) for anio, df in homologados.items()}
for anio, df in hmo.items():
    print(anio, "filas de Hermosillo:", len(df))  # se espera 1 por año

# pd.concat junta las tres filas en una tabla; droplevel(1) quita el número de fila original
# y deja solo el año; .T voltea la tabla para ver los años como columnas
hmo_tabla = pd.concat(hmo, names=["anio", None]).droplevel(1)
hmo_tabla.T

### Verificación: ¿qué se puede comparar entre años?

Se revisan dos cosas:
1. **`IMN` frente a `IM`:** si `IMN / IM` cambia cada año, cada año se normalizó con una escala distinta y el `IMN` **no sirve** para ver cambios en el tiempo.
2. **Cambio 2015 → 2020** de cada indicador, para detectar saltos raros.

**Resultado:**
- `IMN / IM` vale 0.0244 (2010), 0.0184 (2015) y 0.0157 (2020): **cambia cada año**. Por eso el `IMN` de Hermosillo se queda en ~0.946 aunque el `IM` suba. Para ver el cambio en el tiempo se usa `IM`, y `marginacion_hermosillo()` **ya no incluye** el `IMN`.
- `ANALF` y `OVPT` suben muy poco (~0.15 puntos); el resto de las carencias bajan, sobre todo `VHAC` (−9.4).
- `PO2SM` pasa de 20 % a 47 %. Otros municipios de Sonora tienen saltos parecidos, así que probablemente cambió cómo se mide (el salario mínimo subió mucho en 2019–2020). No se debe leer como un empeoramiento real.

In [ ]:
# Columnas numéricas de la tabla de Hermosillo (deja fuera textos como NOM_MUN o GM)
num = hmo_tabla.select_dtypes("number")

# 1) Cociente IMN / IM por año: si fuera el mismo número, la escala sería la misma
print("IMN / IM por año:")
print((num["IMN"] / num["IM"]).round(4))

# 2) Cambio de cada indicador entre 2015 y 2020 (valor 2020 menos valor 2015)
indicadores_censo = ["ANALF", "SBASC", "OVSDE", "OVSEE", "OVSAE", "OVPT", "VHAC", "PL.5000", "PO2SM", "IM"]
cambio = pd.DataFrame({
    "2015": num.loc[2015, indicadores_censo],  # .loc[año, columnas] toma la fila de ese año
    "2020": num.loc[2020, indicadores_censo],
})
cambio["cambio"] = cambio["2020"] - cambio["2015"]  # positivo = subió; negativo = bajó
cambio.round(2)

## Interpolación mensual

`marginacion_hermosillo()` crea un valor por mes:
- **ene-2015:** valor del censo 2015.
- **Entre ene-2015 y ene-2020:** una línea recta entre los dos censos.
- **De ene-2020 en adelante:** valor del censo 2020 (no hay censo posterior para seguir la línea).

La fórmula, para cada indicador y cada mes, es:

```
valor_mes = valor_2015 + (valor_2020 − valor_2015) × fracción
fracción  = días desde ene-2015 / días entre ene-2015 y ene-2020   (se limita a 0–1)
```

Ejemplo con `IM`: en ene-2015 la fracción es 0 (51.51); a mitad del camino (jul-2017) es ~0.5 (~55.97); desde ene-2020 es 1 (60.44).

Se excluyen las claves, los nombres, `GM` (es texto), `POB_TOT` (es población, no un indicador del índice) e `IMN` (no comparable entre años).

**Resultado:** 72 meses × 10 indicadores. La comprobación confirma que ene-2015 coincide con el censo 2015 y que todo 2020 coincide con el censo 2020.

In [ ]:
# Tabla mensual de marginación de Hermosillo (la que entra al panel)
marg_mensual = marginacion_hermosillo()
# Lista de indicadores: todas las columnas menos la fecha
indicadores = [c for c in marg_mensual.columns if c != "fecha_mes"]
print(f"{len(marg_mensual)} meses | indicadores interpolados ({len(indicadores)}):", indicadores)

# Comprobación: ene-2015 debe ser igual al censo 2015, y dic-2020 igual al censo 2020
ene_2015 = marg_mensual.iloc[0][indicadores]  # primera fila (ene-2015)
dic_2020 = marg_mensual.iloc[-1][indicadores]  # última fila (dic-2020)
# (a - b).abs().max() = la mayor diferencia; < 1e-9 significa "prácticamente cero"
print("ene-2015 = censo 2015:", (ene_2015 - num.loc[2015, indicadores]).abs().max() < 1e-9)
print("dic-2020 = censo 2020:", (dic_2020 - num.loc[2020, indicadores]).abs().max() < 1e-9)

### Gráfica del IM mensual

Se dibuja el `IM` interpolado y se marcan con puntos los valores reales de los censos.

**Cómo leerla:** una línea recta que sube de 2015 a 2020 y se queda plana durante 2020. Los puntos deben caer sobre la línea.

In [ ]:
col = "IM"  # se grafica IM porque es el indicador comparable entre años (IMN no)
fig, ax = plt.subplots(figsize=(10, 3.5))  # figura de 10 x 3.5 pulgadas
ax.plot(marg_mensual["fecha_mes"], marg_mensual[col], label="interpolado")  # línea mensual
# Puntos de los censos 2015 y 2020, ubicados en enero de cada año
anios_panel = [a for a in (2015, 2020) if PANEL_START_YEAR <= a <= PANEL_END_YEAR]
ax.scatter(pd.to_datetime([f"{a}-01-01" for a in anios_panel]), num.loc[anios_panel, col],
           color="red", zorder=3, label="censo")  # zorder=3 dibuja los puntos encima de la línea
ax.set_ylabel(col)
ax.set_title(f"{col} mensual interpolado — Hermosillo")
ax.legend()
plt.tight_layout()
plt.show()

# TODO: si se amplía la ventana a 2010, interpolar por tramos 2010→2015→2020
# con los tres censos (requiere ajustar panel.marginacion_hermosillo()).

## DataFrame final

Primeras 5 filas de la tabla mensual (72 meses × 11 columnas: `fecha_mes` + 10 indicadores). Es la que se guarda en la celda siguiente y la que se une al panel por `fecha_mes`.

In [ ]:
pd.set_option("display.max_columns", None)  # mostrar todas las columnas
print("Dimensiones:", marg_mensual.shape)  # (filas, columnas)
# primeras 5 filas; round redondea a 3 decimales solo los indicadores (no la fecha)
marg_mensual.head().round({c: 3 for c in indicadores})

## Guardar salida

Guarda la tabla como parquet en `data/processed/`, que es lo que lee el panel (`03_eda/3.0`): hay que ejecutar esta celda antes. La copia en CSV para revisar en Excel está comentada; para guardarla, quita el `#`.

In [ ]:
# TODO: guardar una vez validada la interpolación
# index=False evita guardar el número de fila como columna extra
# marg_mensual.to_csv(MARG_DIR / "marginacion_hermosillo_mensual.csv", index=False)

# Para el panel: build_panel_monthly() y build_panel_weekly() leen este parquet de data/processed/
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
marg_mensual.to_parquet(PROCESSED_DATA_DIR / "marginacion_hermosillo.parquet", index=False)

## Decisiones de procesamiento

Si una decisión cambia el panel, se implementa en `sonora_health_climate_eda/panel.py`.

| Tema | Decisión | Por qué |
|------|----------|---------|
| Marginación mensual | Interpolación lineal 2015 → 2020, constante después | Solo hay censos; se supone un cambio gradual (limitación documentada) |
| `POB_TOT`, claves, nombres y `GM` | Se excluyen de la interpolación | No son indicadores numéricos del índice |
| `IMN` | **Se elimina** del panel (`marginacion_hermosillo()` lo excluye); usar `IM` | Cada año se normaliza con una escala distinta (`IMN / IM` cambia), así que parece que la marginación no cambia |
| `PO2SM` 2020 | Sin cambios, pero no interpretar el salto como tendencia | Probable cambio de medición; afecta a todo Sonora |
| Censo 2010 | No se usa (fuera de la ventana 2015–2020) | Solo haría falta si se amplía el panel a 2010 |